[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch01-introduction/01-dinh-luat-sat-gpt3.ipynb)

# Định luật sắt: GPT-3 trên 1,024 bộ tăng tốc, và số hạng nào chi phối

Chương 1 gọi định luật sắt (iron law) của hệ thống học máy là xương sống toán học của cả bộ sách. Nó tách
tổng thời gian thực thi $T$ thành ba số hạng:

$$T = \frac{D_\text{vol}}{\text{BW}} + \frac{O}{R_\text{peak}\cdot\eta_\text{hw}} + L_\text{lat}$$

khối lượng dữ liệu chia cho băng thông (bandwidth), số phép toán chia cho tích của thông lượng đỉnh và mức
sử dụng (utilization) phần cứng, và độ trễ (latency) cố định. Sổ tay này viết định luật đó thành vài hàm
Python, làm lại phép tính nháp duy nhất của chương, rồi đi xa hơn: quét từng biến, hỏi số hạng nào đang
chi phối, và đo một trường hợp trên chính CPU của bạn.

**Bạn sẽ làm:**
1. đổi 8.5 tỉ lượt tìm kiếm mỗi ngày ra lượt mỗi giây;
2. ước lượng số phép tính huấn luyện GPT-3 từ số tham số và số token;
3. làm lại Napkin Math 1.1: GPT-3 trên một cụm 1,024 bộ tăng tốc (accelerator), rồi quét số bộ tăng tốc,
   thông lượng đỉnh và mức sử dụng;
4. đặt AlexNet, GPT-3 và GPT-4 lên cùng một thang GPU-ngày;
5. tìm số hạng chi phối ở ba nơi: một job huấn luyện, một đường ống suy luận 130 ms, và một tầng tuyến
   tính đo trên CPU;
6. tính thuế năng lượng theo kích thước batch (batch size), ở FP16 và INT8;
7. đặt lợi tức trên tính toán (return on compute, RoC) cạnh định luật sắt.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định, như các sổ tay khác
GIAY_NGAY = 24 * 3600
ZETTA = 1e21  # 1 zettaFLOP = 10²¹ FLOP
TERA = 1e12

## 1. Quy mô: 8.5 tỉ lượt mỗi ngày

Chương mở đầu bằng hai nghĩa vụ song hành: hành vi học được phải đáng tin, và máy phải tạo ra hành vi đó
trong giới hạn thời gian, bộ nhớ, năng lượng và chi phí. Để cảm được vế thứ hai, hãy đổi con số của chương
về đơn vị một giây.

In [ ]:
LUOT_NGAY = sach(8.5e9, nguon="Google xử lý 8,5 tỉ lượt tìm kiếm mỗi ngày")
luot_giay = LUOT_NGAY / GIAY_NGAY
print(f"{LUOT_NGAY:,.0f} lượt mỗi ngày = {luot_giay:,.0f} lượt mỗi giây, tính trung bình cả ngày")

Gần một trăm nghìn truy vấn mỗi giây, ngày này qua ngày khác, và mỗi truy vấn đều có mục tiêu độ trễ riêng.
Ở quy mô đó, một phần nghìn giây tiết kiệm được trên mỗi truy vấn là gần một trăm giây máy mỗi giây. Phần
còn lại của sổ tay hỏi: thời gian đó đi đâu?

## 2. GPT-3 cần bao nhiêu phép tính?

Chương nêu GPT-3 có 175 tỉ tham số, huấn luyện trên khoảng 300 tỉ token, và "theo cách xấp xỉ huấn luyện
dày đặc của sách" thì cần khoảng 314 zettaFLOP. Xấp xỉ đó (bài 8 viết nó thành $O \approx 6\cdot P\cdot D$)
đếm như sau: với mỗi tham số và mỗi token, lượt truyền xuôi tốn một phép nhân-cộng, tức 2 phép tính dấu
phẩy động (FLOP), còn lượt truyền ngược tốn khoảng gấp đôi, tức 4 FLOP.

**Dự đoán:** $6 \times 175 \text{ tỉ} \times 300 \text{ tỉ}$ là khoảng bao nhiêu zettaFLOP?

In [ ]:
THAM_SO = sach(175e9, nguon="GPT-3 175 tỉ tham số")
TOKEN = sach(300e9, nguon="~300 tỉ token", tol=5e9)
FLOP_MOI_THAM_SO_TOKEN = 2 + 4  # xuôi: một phép nhân-cộng = 2 FLOP; ngược: khoảng gấp đôi

O_uoc_luong = FLOP_MOI_THAM_SO_TOKEN * THAM_SO * TOKEN
print(f"O ≈ 6 × {THAM_SO:.3g} × {TOKEN:.3g} = {O_uoc_luong:.3g} FLOP = {O_uoc_luong / ZETTA:.0f} zettaFLOP")
theo_sach("phép tính huấn luyện GPT-3 (zettaFLOP)", O_uoc_luong / ZETTA, sach=314, nguon="~314 zettaFLOP", tol=2)

Phép nhân cho 315, sách ghi khoảng 314: cả 175 tỉ lẫn 300 tỉ đều là số đã làm tròn, nên lệch nhau chưa tới
1 %. Từ đây sổ tay dùng đúng con số của Napkin Math 1.1, $3.14\times10^{23}$ FLOP, tức 314 zettaFLOP.

## 3. Napkin Math 1.1: huấn luyện GPT-3 mất bao lâu?

Bài toán của chương: huấn luyện một mô hình cỡ GPT-3 trên một cụm tham chiếu có 1,024 bộ tăng tốc thì mất
bao lâu? Mỗi bộ tăng tốc có thông lượng đỉnh 312 TFLOP/s, và mức sử dụng phần cứng khoảng 45 %, mức chương
gọi là điển hình cho huấn luyện phân tán quy mô lớn. Napkin Math chỉ giữ số hạng tính toán của định luật
sắt, nhân thêm số bộ tăng tốc vào mẫu số:

$$T_\text{train} \approx \frac{O}{N_\text{accel}\cdot R_\text{peak}\cdot\eta_\text{hw}}$$

**Dự đoán:** vài giờ, vài ngày, vài tuần, hay vài tháng?

In [ ]:
O_GPT3 = sach(3.14, trich=r"\approx 3.14 \times 10^{23}", tol=0.005) * 1e23
R_PEAK = sach(312, trich="312 TFLOP/s") * TERA
ETA = sach(45, trich="≈ 45 percent (typical for large-scale distributed training)", tol=0.5) / 100
N_ACC = sach(1024, trich="1,024 accelerators")


def ngay_huan_luyen(O, N, R, eta):
    """Số hạng tính toán của định luật sắt, chia cho N bộ tăng tốc, đổi ra ngày."""
    return O / (N * R * eta) / GIAY_NGAY


T45 = ngay_huan_luyen(O_GPT3, N_ACC, R_PEAK, ETA)
print(f"{O_GPT3:.3g} / ({N_ACC:,} × {R_PEAK:.3g} × {ETA}) = {T45 * GIAY_NGAY:.3g} s = {T45:.1f} ngày")
theo_sach("thời gian huấn luyện ở mức sử dụng 45 % (ngày)", T45, sach=25, trich="Result: 25 days")

Chương đi thêm một bước: nếu lập lịch tốt hơn và thực thi hiệu quả hơn đưa mức sử dụng từ 45 % lên 60 %
thì sao?

In [ ]:
ETA_TOT = sach(60, trich="45 percent to 60 percent") / 100
T60 = ngay_huan_luyen(O_GPT3, N_ACC, R_PEAK, ETA_TOT)
theo_sach("thời gian huấn luyện ở mức sử dụng 60 % (ngày)", T60, sach=19, trich="training time drops to 19 days")
theo_sach("số ngày tiết kiệm", T45 - T60, sach=6, trich="saving 6 days of expensive compute time")

### Ba biến ở mẫu số, ba cái giá khác nhau

$N_\text{accel}$, $R_\text{peak}$ và $\eta_\text{hw}$ đứng cạnh nhau ở mẫu số, nên về thời gian chúng đổi
chỗ được cho nhau: gấp đôi bất kỳ biến nào cũng chia đôi số ngày.

**Dự đoán:** về **chi phí** thì sao? Gấp đôi số bộ tăng tốc có làm giảm tổng số bộ tăng tốc-ngày phải trả
không?

In [ ]:
cac_N = np.array([256, 512, 1024, 2048, 4096, 8192])
cac_eta = np.linspace(0.2, 0.9, 50)


def bo_tang_toc_ngay(O, N, R, eta):
    return N * ngay_huan_luyen(O, N, R, eta)


# bộ tăng tốc-ngày không phụ thuộc N: N ở tử số triệt tiêu N ở mẫu số
gia = [bo_tang_toc_ngay(O_GPT3, N, R_PEAK, ETA) for N in cac_N]
assert all(math.isclose(g, gia[0]) for g in gia)
print(f"ở mức sử dụng 45 %: {gia[0]:,.0f} bộ tăng tốc-ngày, với mọi N")
print(f"ở mức sử dụng 60 %: {bo_tang_toc_ngay(O_GPT3, N_ACC, R_PEAK, ETA_TOT):,.0f} bộ tăng tốc-ngày")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
for N, mau in zip((512, 1024, 2048), MAU):
    ax1.plot(100 * cac_eta, ngay_huan_luyen(O_GPT3, N, R_PEAK, cac_eta), color=mau, lw=2, label=f"{N:,} bộ tăng tốc")
for eta, T, lech in ((ETA, T45, (10, 12)), (ETA_TOT, T60, (10, 12))):
    ax1.plot([100 * eta], [T], "o", color="k", ms=6, zorder=3)
    ax1.annotate(f"{100 * eta:.0f} %: {T:.0f} ngày", (100 * eta, T), xytext=lech, textcoords="offset points", fontsize=9)
ax1.set_xlabel("mức sử dụng phần cứng η (%)")
ax1.set_ylabel("số ngày huấn luyện")
ax1.set_title("Cùng 3.14×10²³ FLOP, 312 TFLOP/s mỗi bộ tăng tốc")
ax1.set_ylim(0, 120)
ax1.legend(loc="upper right", fontsize=8)

ngay_N = [ngay_huan_luyen(O_GPT3, N, R_PEAK, ETA) for N in cac_N]
ax2.plot(cac_N, ngay_N, "o-", color=MAU[0], lw=2)
ax2.set_xscale("log", base=2)
ax2.set_yscale("log")
ax2.set_yticks([3, 6, 12, 25, 50, 100], ["3", "6", "12", "25", "50", "100"])
ax2.minorticks_off()
ax2.annotate(f"{N_ACC:,}: {T45:.0f} ngày", (N_ACC, T45), xytext=(8, 6), textcoords="offset points", fontsize=9,
             color=MAU[0])
ax2.set_xlabel("số bộ tăng tốc N (thang log)")
ax2.set_ylabel("số ngày huấn luyện (thang log)")
ax2.set_title("Thêm máy rút ngắn thời gian")
ax2.set_xticks(cac_N, [f"{N:,}" for N in cac_N])

# hai thước đo khác đơn vị nằm ở hai đồ thị riêng, không chung một khung hai trục
ax3.plot(cac_N, [g / 1000 for g in gia], "s--", color=MAU[1], lw=2, label="η = 45 %")
ax3.plot(cac_N, [bo_tang_toc_ngay(O_GPT3, N, R_PEAK, ETA_TOT) / 1000 for N in cac_N], "s:", color=MAU[2], lw=2,
         label="η = 60 %")
ax3.set_xscale("log", base=2)
ax3.minorticks_off()
ax3.set_xticks(cac_N, [f"{N:,}" for N in cac_N])
ax3.set_ylim(0, 35)
ax3.set_xlabel("số bộ tăng tốc N (thang log)")
ax3.set_ylabel("nghìn bộ tăng tốc-ngày")
ax3.set_title("…nhưng không giảm chi phí")
ax3.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Hai đồ thị bên phải nói hai chuyện. Thêm bộ tăng tốc rút ngắn thời gian lịch, nhưng tổng số bộ tăng tốc-ngày, thứ
mà hoá đơn tính tiền, đứng yên: $N$ ở tử số triệt tiêu $N$ ở mẫu số. Nâng mức sử dụng thì giảm cả hai. Đó là
lý do chương gọi 6 ngày tiết kiệm được là 6 ngày tính toán đắt đỏ: chúng là máy không phải thuê, chứ không
chỉ là kết quả về sớm hơn.

Phép tính này giả định $\eta_\text{hw}$ giữ nguyên khi $N$ đổi. Phần *Thử thêm* cho bạn bỏ giả định đó.

## 4. Ba mốc trên một thang GPU-ngày

Bảng 3 của chương đặt mỗi thời kỳ cạnh nền phần cứng của nó. AlexNet huấn luyện 6 ngày trên 2 GPU. Mô hình
nền cỡ GPT-4 được ước lượng khoảng 2.5 triệu GPU-ngày tham chiếu, "cỡ 25,000 GPU tham chiếu chạy 90 ngày".
GPT-3 của Napkin Math nằm ở giữa.

In [ ]:
ALEX_NGAY = sach(6, nguon="trong 6 ngày trên 2 GPU")
ALEX_GPU = sach(2, nguon="trong 6 ngày trên 2 GPU")
GPT4_GPU_NGAY = sach(2.5e6, nguon="~2,5 triệu GPU-ngày tham chiếu", tol=0.05e6)
GPT4_GPU = sach(25_000, trich="on the order of 25,000 reference GPUs run for 90 days")
GPT4_NGAY = sach(90, trich="on the order of 25,000 reference GPUs run for 90 days")

moc = {"AlexNet (2012)": ALEX_NGAY * ALEX_GPU,
       "GPT-3, Napkin Math 1.1": N_ACC * T45,
       "cỡ GPT-4 (2023)": GPT4_GPU_NGAY}
for ten, v in moc.items():
    print(f"  {ten:<24} {v:>12,.0f} GPU-ngày")
print(f"25,000 GPU × 90 ngày = {GPT4_GPU * GPT4_NGAY:,.0f} GPU-ngày; bảng ghi khoảng {GPT4_GPU_NGAY:,.0f}")

Hai cách ghi của bảng 3 cho mốc GPT-4 không khớp nhau từng chữ số: 25,000 nhân 90 là 2.25 triệu, còn bảng
ghi khoảng 2.5 triệu. Cả hai là ước lượng minh hoạ, cùng bậc độ lớn, và bảng tự nói "cỡ". Sổ tay in cả hai,
không khẳng định bên nào.

Chương còn nói lượng tính toán từ AlexNet đến mốc cỡ GPT-4 tăng khoảng 7 bậc độ lớn. Số GPU-ngày thì tăng
ít bậc hơn. Phần chênh lệch là phần mà mỗi GPU-ngày đã làm được nhiều việc hơn.

In [ ]:
BAC_TINH_TOAN = sach(7, nguon="khoảng cách tổng ~7 bậc độ lớn", tol=0.5)
bac_gpu_ngay = math.log10(GPT4_GPU_NGAY / (ALEX_NGAY * ALEX_GPU))
print(f"GPU-ngày tăng {GPT4_GPU_NGAY / (ALEX_NGAY * ALEX_GPU):,.0f} lần, tức {bac_gpu_ngay:.1f} bậc độ lớn")
print(f"tính toán tăng khoảng {BAC_TINH_TOAN} bậc: mỗi GPU-ngày làm nhiều việc hơn khoảng "
      f"10^{BAC_TINH_TOAN - bac_gpu_ngay:.1f} ≈ {10 ** (BAC_TINH_TOAN - bac_gpu_ngay):.0f} lần")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ten_moc, gia_moc = list(moc), list(moc.values())
ax.barh(ten_moc, gia_moc, color=MAU[:3], height=0.55)
ax.set_xscale("log")
ax.set_xlim(1, 3e8)
for i, v in enumerate(gia_moc):
    ax.text(v * 1.4, i, f"{v:,.0f}", va="center", fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("GPU-ngày (thang log; mỗi mốc một loại GPU tham chiếu khác nhau)")
ax.set_title("Bảng 3 và Napkin Math 1.1 trên một trục")
plt.tight_layout()
plt.show()

Con số 10 mũ 1.7 ở trên chỉ đúng ở bậc độ lớn: nó được suy ra từ hai ước lượng minh hoạ, đo bằng hai loại
"GPU tham chiếu" khác nhau.

Quy mô đó còn có giá bằng điện. Chương dẫn ước lượng khoảng 1.3 GWh cho mô hình cỡ GPT-3. Nếu, giả định,
lượng điện đó được tiêu hết trong 25 ngày của Napkin Math, công suất trung bình là:

In [ ]:
GWH_GPT3 = sach(1.3, nguon="~1,3 GWh điện cho mô hình cỡ GPT-3", tol=0.05)
mw = GWH_GPT3 * 1e3 / (T45 * 24)  # GWh → MWh, chia cho số giờ
print(f"{GWH_GPT3} GWh / ({T45:.1f} ngày × 24 h) = {mw:.1f} MW trung bình")

Vài mê-ga-oát, liên tục trong gần bốn tuần, đúng cỡ "những cụm máy nhiều mê-ga-oát" mà chương nhắc tới. Đây
là phép ghép hai ước lượng khác nguồn: 1.3 GWh là ước lượng cho một mô hình cỡ GPT-3, còn cụm 1,024 bộ tăng
tốc là kịch bản minh hoạ của Napkin Math.

## 5. Số hạng nào đang chi phối?

Napkin Math bỏ qua hai số hạng còn lại. Checkpoint 1.2 của chương hỏi đúng điều cần kiểm: gấp đôi
$R_\text{peak}$ thì số hạng nào cải thiện, và vì sao hai số hạng kia đứng yên? Câu trả lời phụ thuộc vào
việc số hạng tính toán chiếm bao nhiêu phần của $T$.

Định luật sắt có hai dạng. Khi các giai đoạn chạy lần lượt, ba số hạng cộng lại. Khi di chuyển dữ liệu chồng
lấn được với tính toán, $T$ chỉ còn ít nhất bằng số hạng lớn hơn trong hai số hạng đó, cộng độ trễ cố định:

$$T_\text{pipelined} \ge \max\left(\frac{D_\text{vol}}{\text{BW}},\ \frac{O}{R_\text{peak}\cdot\eta_\text{hw}}\right) + L_\text{lat}$$

In [ ]:
def dinh_luat_sat(D_vol, BW, O, R_peak, eta_hw, L_lat=0.0):
    """Ba số hạng của định luật sắt (giây), dạng cộng và dạng chồng lấn."""
    du_lieu = D_vol / BW
    tinh_toan = O / (R_peak * eta_hw)
    return {"dữ liệu": du_lieu, "tính toán": tinh_toan, "độ trễ": L_lat,
            "cộng": du_lieu + tinh_toan + L_lat,
            "chồng lấn": max(du_lieu, tinh_toan) + L_lat}


def tang_toc(goc, **doi):
    """T trước chia T sau, khi một vài biến của định luật sắt đổi, ở dạng cộng."""
    return dinh_luat_sat(**goc)["cộng"] / dinh_luat_sat(**{**goc, **doi})["cộng"]

### (a) Một job huấn luyện

Số hạng dữ liệu của một job huấn luyện có nhiều phần. Ở đây chỉ tính một phần dễ thấy nhất: đọc tập token
một lần từ lưu trữ. Chương ghi tập token của GPT-3 chiếm khoảng 420 GB. Băng thông lưu trữ là giả định của
sổ tay, cố ý chọn thấp: 1 GB/s.

In [ ]:
D_TAP_TOKEN = sach(420, nguon="tập token ~420 GB", tol=5) * 1e9
BW_LUU_TRU = 1e9  # giả định: 1 GB/s, cố ý thấp
job = dict(D_vol=D_TAP_TOKEN, BW=BW_LUU_TRU, O=O_GPT3, R_peak=N_ACC * R_PEAK, eta_hw=ETA)
so_hang = dinh_luat_sat(**job)
print(f"đọc tập token một lần: {so_hang['dữ liệu'] / 60:.0f} phút · tính toán: {so_hang['tính toán'] / GIAY_NGAY:.1f} ngày")
print(f"phần thời gian của số hạng tính toán: {so_hang['tính toán'] / so_hang['cộng']:.4%}")
print(f"gấp đôi R_peak: nhanh hơn {tang_toc(job, R_peak=2 * N_ACC * R_PEAK):.4f} lần")
print(f"gấp đôi băng thông lưu trữ: nhanh hơn {tang_toc(job, BW=2 * BW_LUU_TRU):.4f} lần")

Với job này, số hạng tính toán chiếm gần như toàn bộ thời gian, nên gấp đôi thông lượng đỉnh gần như chia
đôi $T$, còn gấp đôi băng thông lưu trữ gần như không đổi gì. Đó là lý do Napkin Math được phép bỏ hai số
hạng kia. Lưu ý rằng số hạng dữ liệu thật của huấn luyện còn gồm trọng số và giá trị kích hoạt (activation)
đi qua bộ nhớ ở mỗi bước, lớn hơn nhiều lần một lượt đọc tập token; bài 2 và bài 8 tính phần đó.

### (b) Một đường ống suy luận: ngộ nhận về tối ưu từng thành phần

Phần ngộ nhận và cạm bẫy của chương kể một nhóm giảm thời gian suy luận từ 45 ms xuống 15 ms và chờ tổng thời
gian giảm theo. Nhưng tiền xử lý tốn 60 ms và hậu xử lý thêm 25 ms.

**Dự đoán:** suy luận nhanh gấp 3 lần thì cả đường ống nhanh hơn bao nhiêu phần trăm?

In [ ]:
SUY_LUAN = sach(45, nguon="45 ms → 15 ms suy luận")
SUY_LUAN_MOI = sach(15, nguon="45 ms → 15 ms suy luận")
TIEN_XU_LY = sach(60, nguon="tiền xử lý 60 ms")
HAU_XU_LY = sach(25, nguon="hậu xử lý 25 ms")

truoc = TIEN_XU_LY + SUY_LUAN + HAU_XU_LY
sau = TIEN_XU_LY + SUY_LUAN_MOI + HAU_XU_LY
theo_sach("tổng độ trễ trước (ms)", truoc, sach=130, nguon="130 ms → 100 ms")
theo_sach("tổng độ trễ sau (ms)", sau, sach=100, nguon="130 ms → 100 ms")
theo_sach("cải thiện thật (%)", 100 * (truoc - sau) / truoc, sach=23, nguon="23 % chứ không phải 67 %")
theo_sach("cải thiện mong đợi (%)", 100 * (SUY_LUAN - SUY_LUAN_MOI) / SUY_LUAN, sach=67, nguon="23 % chứ không phải 67 %")
print(f"giới hạn: suy luận tốn 0 ms thì tổng còn {TIEN_XU_LY + HAU_XU_LY} ms, "
      f"cải thiện tối đa {100 * SUY_LUAN / truoc:.1f} %")

Định luật Amdahl (Amdahl's law) tổng quát hoá điều này: nếu phần $f$ của tổng thời gian nhanh lên $k$ lần, cả
hệ thống nhanh lên $1/\big((1-f) + f/k\big)$ lần. Đồ thị bên trái quét mức tăng tốc của riêng bước suy luận;
đồ thị bên phải trả lời câu hỏi của Checkpoint 1.2 cho mọi tải công việc (workload): gấp đôi tốc độ của một
số hạng đem lại bao nhiêu, theo phần thời gian số hạng đó đang chiếm.

In [ ]:
k = np.array([1, 1.5, 2, 3, 5, 10, 30, 100])
tong = TIEN_XU_LY + HAU_XU_LY + SUY_LUAN / k
f = np.linspace(0, 1, 101)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.semilogx(k, tong, "o-", color=MAU[0], lw=2)
ax1.axhline(TIEN_XU_LY + HAU_XU_LY, color="0.5", ls="--", lw=1)
ax1.text(1.05, TIEN_XU_LY + HAU_XU_LY - 4, f"tiền xử lý + hậu xử lý = {TIEN_XU_LY + HAU_XU_LY} ms", fontsize=8, color="0.3",
         va="top")
ax1.annotate(f"nhanh 3 lần: {sau} ms (−23 %)", (3, sau), xytext=(14, 14), textcoords="offset points", fontsize=9,
             arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax1.set_xticks(k, [f"{v:g}" for v in k])
ax1.set_ylim(70, 140)
ax1.set_xlabel("suy luận nhanh lên bao nhiêu lần (thang log)")
ax1.set_ylabel("tổng độ trễ đầu-cuối (ms)")
ax1.set_title("Đường ống 60 + 45 + 25 ms")

ax2.plot(100 * f, 1 / ((1 - f) + f / 2), color=MAU[1], lw=2)
for ten, ff, lech in (("suy luận trong\nđường ống 130 ms", SUY_LUAN / truoc, (10, -30)),
                      ("job huấn luyện\nở phần (a)", so_hang["tính toán"] / so_hang["cộng"], (-80, -14))):
    ax2.plot([100 * ff], [1 / ((1 - ff) + ff / 2)], "o", color="k", ms=6)
    ax2.annotate(ten, (100 * ff, 1 / ((1 - ff) + ff / 2)), xytext=lech, textcoords="offset points", fontsize=8)
ax2.set_xlabel("phần thời gian của số hạng được tăng tốc (%)")
ax2.set_ylabel("cả hệ thống nhanh hơn (lần)")
ax2.set_title("Gấp đôi tốc độ của một số hạng")
plt.tight_layout()
plt.show()

Kết luận của chương: cải thiện ở một thành phần không quyết định cải thiện đầu-cuối; kết quả phụ thuộc vào
phần đường đi mà thành phần đó chiếm. Một đòn bẩy chỉ đáng giá theo phần thời gian của số hạng mà nó tác động.

### (c) Một tầng tuyến tính, đo trên CPU của bạn

Chương dùng hai mô hình đại diện (lighthouse model) để cho thấy cùng một phương trình đưa ra hai chẩn đoán.
ResNet-50 theo lô dùng lại cùng một bộ tham số rất nhiều lần, nên số hạng tính toán chi phối. GPT-2 khi sinh
token nạp từng tham số và chỉ dùng nó một lần, nên số hạng băng thông chi phối. Vì vậy gấp đôi thông lượng
đỉnh giúp ResNet-50 theo lô nhưng gần như không giúp GPT-2 khi sinh token.

Một tầng tuyến tính có cả hai chế độ đó. Với batch 1, mỗi trọng số đọc từ bộ nhớ chỉ dùng cho một phép
nhân-cộng. Với batch 256, mỗi trọng số đọc một lần mà dùng 256 lần. Sổ tay chọn tầng 4096 × 4096 ở FP32: 64 MB
trọng số, lớn hơn bộ nhớ đệm của hầu hết CPU, nên mỗi lượt phải đọc lại trọng số từ DRAM. Số FLOP trên mỗi
byte phải di chuyển, tức mật độ tính toán (arithmetic intensity), của hai trường hợp khác nhau hơn hai trăm lần:

In [ ]:
D = 4096                       # tầng D × D, FP32, giả định của sổ tay
BYTE = 4


def flop_va_byte(B):
    flop = 2 * B * D * D                      # mỗi phần tử ra: D phép nhân-cộng
    byte = BYTE * (D * D + B * D + B * D)     # đọc trọng số một lần, đọc đầu vào, ghi đầu ra
    return flop, byte


for B in (1, 256):
    flop, byte = flop_va_byte(B)
    print(f"batch {B:>3}: {flop / 1e6:8.1f} MFLOP, {byte / 1e6:5.1f} MB, {flop / byte:6.1f} FLOP mỗi byte")
assert math.isclose(flop_va_byte(1)[0] / flop_va_byte(1)[1], 2 * D / (4 * (D + 2)))

**Dự đoán:** CPU của bạn chạy hai trường hợp đó với cùng số FLOP mỗi giây, hay một trường hợp nhanh hơn hẳn?
Nếu ngày mai bạn có một CPU tính nhanh gấp đôi mà bộ nhớ giữ nguyên, trường hợp nào được lợi?

In [ ]:
W = torch.randn(D, D)


def trung_vi_thoi_gian(f, lap=9):
    f()  # chạy khởi động một lần
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


toc_do = {}
with torch.no_grad():
    for B in (1, 256):
        x = torch.randn(B, D)
        t = trung_vi_thoi_gian(lambda: x @ W)
        toc_do[B] = flop_va_byte(B)[0] / t
        do_tren_may(f"batch {B:>3}, thời gian một lượt", 1e3 * t, "ms")
        do_tren_may(f"batch {B:>3}, tốc độ đạt được", toc_do[B] / 1e9, "GFLOP/s")
do_tren_may("batch 256 so với batch 1, FLOP mỗi giây", toc_do[256] / toc_do[1], "lần")

Trên hầu hết các máy, batch 256 đạt số FLOP mỗi giây cao hơn hẳn batch 1, dù chạy cùng phép nhân ma trận trên
cùng con chip. Ở batch 1, CPU phần lớn thời gian chờ trọng số đi từ DRAM tới, như GPT-2 khi sinh token: một CPU
tính nhanh gấp đôi chỉ rút được phần nhỏ của thời gian đó. Ở batch 256, nó phần lớn thời gian tính, như
ResNet-50 theo lô, và đó là trường hợp được lợi từ thông lượng đỉnh. Sổ tay 04 của bài 2 quét đủ các batch và dựng lại định
luật sắt từ hai con số đo được trên máy bạn.

## 6. Thuế năng lượng

Định luật sắt nói về thời gian. Chương thêm một số hạng thứ tư cho năng lượng:

$$E_\text{total} \approx D_\text{vol}\times E_\text{move} + O\times E_\text{compute}$$

Với các hằng số năng lượng sách dùng, chuyển 1 byte từ DRAM ngoài chip tốn khoảng 145.5 lần năng lượng một
phép nhân FP16, và khoảng 800 lần một phép nhân INT8. Sổ tay đo năng lượng bằng đơn vị "một byte đọc từ
DRAM", nên không cần biết con số tuyệt đối.

Chương cũng dặn: tính trên mỗi lần truy cập thì di chuyển đắt hơn hẳn, nhưng một tải công việc trọn vẹn có bị
di chuyển chi phối hay không còn tuỳ số lần truyền và độ rộng mỗi lần, so với số phép toán của nó. Cùng tầng
tuyến tính ở trên, giờ ở FP16 (2 byte mỗi số) và INT8 (1 byte), với trọng số đọc từ DRAM một lần cho mỗi
batch, cho thấy điều đó.

**Dự đoán:** ở batch 1, phần năng lượng dành cho di chuyển dữ liệu là bao nhiêu phần trăm? Ở batch nào thì
phép nhân bắt đầu tốn nhiều năng lượng hơn việc di chuyển?

In [ ]:
DRAM_SO_VOI_FP16 = sach(145.5, nguon="~145,5× năng lượng một phép nhân FP16", tol=0.05)
DRAM_SO_VOI_INT8 = sach(800, nguon="~800× một phép nhân INT8", tol=1)
D_TANG = 4096  # tầng 4096 × 4096, giả định của sổ tay


def nang_luong(B, byte_moi_so, dram_so_voi_phep_nhan):
    """(di chuyển, tính toán), đơn vị: năng lượng của một byte đọc từ DRAM."""
    di_chuyen = byte_moi_so * (D_TANG * D_TANG + 2 * B * D_TANG)   # trọng số + đầu vào + đầu ra
    tinh_toan = B * D_TANG * D_TANG / dram_so_voi_phep_nhan         # mỗi phép nhân là 1/145.5 hay 1/800 byte
    return di_chuyen, tinh_toan


cac_B = 2 ** np.arange(0, 13)
bang = []
for B in (1, 32, 256, 1024):
    m16, c16 = nang_luong(B, 2, DRAM_SO_VOI_FP16)
    m8, c8 = nang_luong(B, 1, DRAM_SO_VOI_INT8)
    bang.append({"batch": B, "FP16: di chuyển (%)": 100 * m16 / (m16 + c16),
                 "INT8: di chuyển (%)": 100 * m8 / (m8 + c8),
                 "INT8 so với FP16, tổng năng lượng": (m8 + c8) / (m16 + c16)})
print(pd.DataFrame(bang).round(3).to_string(index=False))


def hoa_von(byte_moi_so, dram_so_voi_phep_nhan):
    """Batch nhỏ nhất mà phép nhân tốn năng lượng hơn di chuyển."""
    B = 1
    while True:
        m, c = nang_luong(B, byte_moi_so, dram_so_voi_phep_nhan)
        if c >= m:
            return B
        B += 1


B16, B8 = hoa_von(2, DRAM_SO_VOI_FP16), hoa_von(1, DRAM_SO_VOI_INT8)
print(f"batch hoà vốn: FP16 {B16}, INT8 {B8}")

fig, ax = plt.subplots(figsize=(7.5, 4))
for byte_moi_so, he_so, ten, mau, B_hv in ((2, DRAM_SO_VOI_FP16, "FP16 (145.5×)", MAU[0], B16),
                                           (1, DRAM_SO_VOI_INT8, "INT8 (800×)", MAU[1], B8)):
    m, c = nang_luong(cac_B, byte_moi_so, he_so)
    ax.semilogx(cac_B, 100 * m / (m + c), "o-", color=mau, lw=2, base=2, label=f"{ten}, hoà vốn ở batch {B_hv:,}")
    ax.plot([B_hv], [50], "X", color=mau, ms=11, mec="white", zorder=4)
ax.axhline(50, color="0.6", lw=0.8, ls="--")
ax.set_xticks(cac_B, [str(b) for b in cac_B], fontsize=8)
ax.set_ylim(0, 105)
ax.set_xlabel("kích thước batch (thang log)")
ax.set_ylabel("phần năng lượng cho di chuyển dữ liệu (%)")
ax.set_title("Tầng 4096 × 4096, trọng số đọc từ DRAM mỗi batch một lần")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

Ba điều đọc được từ bảng và đồ thị.

* Ở batch 1, gần như toàn bộ năng lượng là di chuyển dữ liệu, ở cả hai độ chính xác. Đó là chế độ của GPT-2
  khi sinh token.
* Cần batch cỡ vài trăm (FP16) hay hơn một nghìn (INT8) thì phép nhân mới tốn ngang việc di chuyển. Nếu bỏ qua
  đầu vào và đầu ra, batch hoà vốn bằng tỉ số năng lượng nhân với số byte mỗi trọng số: 145.5 × 2 = 291 ở FP16,
  800 × 1 = 800 ở INT8. Đầu vào và đầu ra cũng phải đi qua DRAM, nên đẩy cả hai con số lên.
* Ở batch nhỏ, INT8 tiết kiệm gần một nửa năng lượng, và gần như toàn bộ phần tiết kiệm đến từ việc phải di
  chuyển một nửa số byte, không phải từ phép nhân rẻ hơn.

Vì vậy chương kết luận: giảm phần di chuyển dữ liệu không cần thiết có thể cải thiện cùng lúc cả tốc độ lẫn
hiệu quả năng lượng.

## 7. Lợi tức trên tính toán

Cùng những số hạng quyết định thời gian và năng lượng cũng quyết định chi phí. Chương định nghĩa lợi tức trên
tính toán:

$$\text{RoC} = \frac{\Delta\text{Accuracy}}{\Delta\text{Compute Cost}}$$

và cảnh báo rằng một mức tăng 1 điểm phần trăm có thể không qua được phép thử RoC nếu nó đòi số phép toán tăng
10 lần. Đặt ví dụ đó lên cụm của Napkin Math, lấy bộ tăng tốc-ngày làm đơn vị chi phí:

In [ ]:
DIEM = sach(1, nguon="1 điểm phần trăm có thể trượt nếu cần 10× số phép toán")
LAN = sach(10, nguon="1 điểm phần trăm có thể trượt nếu cần 10× số phép toán")

for eta in (ETA, ETA_TOT):
    goc = bo_tang_toc_ngay(O_GPT3, N_ACC, R_PEAK, eta)
    moi = bo_tang_toc_ngay(LAN * O_GPT3, N_ACC, R_PEAK, eta)
    roc = DIEM / (moi - goc)
    print(f"η = {100 * eta:.0f} %: {goc:,.0f} → {moi:,.0f} bộ tăng tốc-ngày, "
          f"{ngay_huan_luyen(LAN * O_GPT3, N_ACC, R_PEAK, eta):.0f} ngày lịch; "
          f"RoC = {roc * 1e6:.2f} điểm mỗi triệu bộ tăng tốc-ngày")
assert math.isclose(bo_tang_toc_ngay(O_GPT3, N_ACC, R_PEAK, ETA) / bo_tang_toc_ngay(O_GPT3, N_ACC, R_PEAK, ETA_TOT),
                    ETA_TOT / ETA)

Một điểm phần trăm, trên cụm này, đổi bằng hơn hai trăm nghìn bộ tăng tốc-ngày, và cả lần huấn luyện kéo dài hơn tám tháng lịch. Có đáng
hay không là câu hỏi kinh tế mà chương để lại cho kỹ sư: nó tuỳ vào giá một bộ tăng tốc-ngày và giá trị của
một điểm độ chính xác cho sản phẩm. Phép thử RoC biến độ chính xác từ một mục tiêu nghiên cứu thành một ngân
sách kỹ thuật.

Nâng mức sử dụng từ 45 % lên 60 % không thêm điểm nào, nhưng thu nhỏ mẫu số của **mọi** thí nghiệm sau đó đi
một phần tư, nên RoC của mọi thí nghiệm tăng gấp 4/3. Chương nói mọi tối ưu ở các chương sau đều nhắm vào
hoặc tử số, hoặc mẫu số của tỉ số này. Sổ tay 03 đo RoC thật trên CPU, với những mạng nhỏ.

## Thử thêm

1. Ở phần 3, bỏ giả định mức sử dụng cố định: giả định `eta = 0.45 * (1024 / N) ** 0.15`, tức mức sử dụng
   giảm dần khi cụm lớn lên. Vẽ lại số ngày và số bộ tăng tốc-ngày theo `cac_N`. Số ngày còn giảm nhanh như trước
   không, và hoá đơn tăng bao nhiêu khi cụm đi từ 1,024 lên 8,192 bộ tăng tốc?
2. Ở phần 5b, thay dòng `TIEN_XU_LY = sach(...)` bằng `TIEN_XU_LY = 20` rồi chạy lại. Suy luận nhanh gấp 3 lần
   giờ đem lại bao nhiêu phần trăm? (Các dấu ✗ khi đó là điều được chờ đợi: sách tính với 60 ms.)
3. Ở phần 6, đổi `D_TANG` thành 512. Batch hoà vốn đổi thế nào, và vì sao đầu vào cùng đầu ra giờ chiếm phần
   lớn hơn trong số byte phải di chuyển?

## Tóm lại

* Napkin Math 1.1: $3.14\times10^{23}$ FLOP trên 1,024 bộ tăng tốc 312 TFLOP/s, ở mức sử dụng 45 %, mất khoảng
  25 ngày; ở 60 % còn 19 ngày.
* Trong số hạng tính toán, $N$, $R_\text{peak}$ và $\eta_\text{hw}$ đổi chỗ được cho nhau về thời gian, nhưng
  thêm máy không giảm số bộ tăng tốc-ngày phải trả, còn nâng mức sử dụng thì có.
* Một đòn bẩy chỉ đáng giá theo phần thời gian của số hạng nó tác động: suy luận nhanh gấp 3 lần chỉ làm đường
  ống 130 ms nhanh hơn 23 %.
* Cùng một tầng tuyến tính có thể nghẽn ở bộ nhớ (batch 1) hoặc ở tính toán (batch lớn), cả về thời gian lẫn
  năng lượng; ở batch nhỏ, năng lượng gần như toàn là di chuyển dữ liệu.
* RoC đặt mức tăng độ chính xác cạnh chi phí của nó; mọi tối ưu đều nhắm vào tử số hoặc mẫu số.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nhập môn hệ thống học máy](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch01-introduction/), dựng từ chương
[*Introduction*](https://mlsysbook.ai/vol1/introduction/introduction.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_01_ml_intro.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.